In [ ]:
# !pip install boto3

In [1]:
import numpy as np
import pandas as pd
import boto3
import os
from dotenv import load_dotenv
from io import StringIO
from gensim.models import Word2Vec
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split

In [21]:
load_dotenv()
s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_DEFAULT_REGION")
)

bucket_name = os.getenv("S3_BUCKET_NAME")
key = "processed/data_cleaned.csv"

s3.download_file(bucket_name, key, "../data/processed/data_cleaned.csv")

In [2]:
data = pd.read_csv("../data/processed/data_cleaned.csv")
data.head()

,text,label
0,"Daniel Greenfield, a Shillman Journalism Fello...",1
1,Google Pinterest Digg Linkedin Reddit Stumbleu...,1
2,U.S. Secretary of State John F. Kerry said Mon...,0
3,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",1
4,It's primary day in New York and front-runners...,0


In [3]:
X = data['text']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [4]:
def tokenize(text):
    return text.lower().split()

sentences = [tokenize(text) for text in X_train]
print(sentences[0])

['bni', 'store', 'nov', '6', '2016', 'like', 'a', 'good', 'little', 'sharia-compliant', 'female,', 'prince', 'charles’', 'wife', 'camilla', 'removes', 'her', 'shoes', 'to', 'enter', 'a', 'mosque', 'in', 'abu', 'dhabi,', 'but', 'the', 'prince', 'of', 'wales', 'keeps', 'his', 'shoes', 'on', 'the', 'prince', 'of', 'wales', 'and', 'duchess', 'of', 'cornwall', 'have', 'visited', 'the', 'spectacular', 'sheikh', 'zaved', 'grand', 'mosque', 'to', 'promote', 'religious', 'tolerance.', '(', 'hah!', ')', 'uk', 'daily', 'mail', 'charles', 'was', 'dressed', 'in', 'a', 'linen', 'suit', 'and', 'striped', 'tie,', 'while', 'camilla', 'wore', 'a', 'blue', 'headscarf,', 'long', 'jacket', 'and', 'trousers.', 'visitors', 'to', 'the', 'mosque', 'must', 'remove', 'their', 'footwear,', 'but', 'charles', 'walked', 'round', 'in', 'black', 'shoes', 'while', 'his', 'wife', 'went', 'barefoot', 'with', 'her', 'head', 'covered.', 'the', 'mosque', 'was', 'established', 'in', '2008', 'and', 'sits', 'at', 'the', 'entra

In [5]:
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2)

In [6]:
class Word2VecTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model):
        self.model = model
        self.dim = model.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        vectors = []
        for text in X:
            words = tokenize(text)
            vecs = [self.model.wv[w] for w in words if w in self.model.wv]
            vectors.append(np.mean(vecs, axis=0) if vecs else np.zeros(self.dim))
        return np.array(vectors)

In [7]:
model = Pipeline([
    ("w2v", Word2VecTransformer(w2v_model)),
    ("clf", LinearSVC(class_weight='balanced'))
])

model.fit(X_train, y_train)

c:\Users\nada amalia\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\nada amalia\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1242: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Pipeline(steps=[('w2v',
                 Word2VecTransformer(model=<gensim.models.word2vec.Word2Vec object at 0x0000021D3FA104A0>)),
                ('clf', LinearSVC(class_weight='balanced'))])

In [8]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Real", "Fake"]))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9021310181531176

Classification Report:
              precision    recall  f1-score   support

        Real       0.90      0.90      0.90       634
        Fake       0.90      0.90      0.90       633

    accuracy                           0.90      1267
   macro avg       0.90      0.90      0.90      1267
weighted avg       0.90      0.90      0.90      1267


Confusion Matrix:
[[573  61]
 [ 63 570]]


In [9]:
def predict_text(text_input, model):
    text_input = str(text_input)
    pred = model.predict([text_input])[0]
    return "Fake" if pred == 1 else "Real"


user_input = input("Input text: ")
hasil = predict_text(user_input, model)

print("Input:", user_input)
print("Label:", hasil)

Input: trump is a president
Label: Real


In [30]:
# !pip install joblib

In [10]:
import joblib
joblib.dump(model, "../outputs/model.pkl")

['../outputs/model.pkl']